In [ ]:
# Goal: Use k-shot voting to predict digit class.

In [ ]:
# Build reference set from x_small, y_small
ref_by_class = {c: np.where(y_small == c)[0] for c in range(10)}

In [ ]:
# Voting function
def predict_digit_by_voting(x_query, k=5, seed=0):
    rng = np.random.default_rng(seed)
    votes = np.zeros(10, dtype=int)

    xq = (x_query / 255.0).reshape(1, 784)

    for c in range(10):
        ref_ids = rng.choice(ref_by_class[c], size=k, replace=False)
        # build pair batch (k, 2, 784)
        batch = np.stack([np.repeat(xq, k, axis=0), x_small[ref_ids]], axis=1)
        preds = model.predict(batch, verbose=0).reshape(-1)
        votes[c] = int((preds > 0.5).sum())

    return votes.argmax(), votes

# quick test
xq_img = xte[0]
pred, votes = predict_digit_by_voting(xq_img, k=5, seed=1)
pred, votes

In [ ]:
# Evaluate on a sample
def eval_voting(n=200, k=5, seed=1):
    rng = np.random.default_rng(seed)
    ids = rng.choice(np.arange(len(xte)), size=n, replace=False)
    correct = 0
    for i in ids:
        pred, _ = predict_digit_by_voting(xte[i], k=k, seed=seed+i)
        correct += int(pred == yte[i])
    return correct / n

for k in [2,3,4,5,6,7]:
    print(k, eval_voting(n=200, k=k))